In [1]:
# ================================================================
# PM2.5 Level Prediction using LSTM (PyTorch)
# with Borderline-SMOTE (class 2,3 only) on train_split.csv
# ================================================================
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

from imblearn.over_sampling import BorderlineSMOTE

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# ---------------- 1) Load dataset ----------------
train_df = pd.read_csv("train_split.csv")
test_df = pd.read_csv("test_split.csv")

# แยก X, y
X_train = train_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_train = train_df["PM2.5_level"].values.astype(np.int64) - 1  # shift to 0-based

X_test = test_df.drop(columns=["PM2.5_level"]).values.astype(np.float32)
y_test = test_df["PM2.5_level"].values.astype(np.int64) - 1

print("Train size:", X_train.shape, " Test size:", X_test.shape)

# เช็คว่า normalize มาละ
train_df.describe()

Train size: (329623, 12)  Test size: (82406, 12)


,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM,wd_sin,wd_cos,PM2.5_level
count,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000,329623.000000
mean,0.102852,0.030595,0.171396,0.223032,0.113731,0.542939,0.469916,0.632490,0.000910,0.130837,0.531205,0.559493,3.511287
std,0.091980,0.043270,0.120707,0.208656,0.111510,0.185810,0.173278,0.190237,0.011691,0.094277,0.353467,0.347421,1.550921
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
25%,0.034000,0.005000,0.076000,0.082000,0.022000,0.374000,0.329000,0.474000,0.000000,0.068000,0.146000,0.146000,2.000000
50%,0.080000,0.013000,0.145000,0.163000,0.088000,0.558000,0.464000,0.640000,0.000000,0.106000,0.500000,0.691000,4.000000
75%,0.143000,0.037000,0.242000,0.286000,0.162000,0.701000,0.606000,0.807000,0.000000,0.167000,0.854000,0.854000,5.000000
max,0.996000,0.822000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.977000,1.000000,1.000000,5.000000


In [3]:
# ---------------- 2) SMOTE + Tomek Links (เฉพาะคลาส 2,3 ⇒ หลัง shift คือ class 1,2)
#     เป้าหมาย = 50% ของคลาสที่มากที่สุดใน TRAIN เท่านั้น ----------------
from imblearn.combine import SMOTETomek
from collections import Counter

before = Counter(y_train)
print("Before SMOTE+Tomek (train):", before)

max_class_count = max(before.values())
target_size = int(max_class_count * 0.5)

sampling_strategy = {}
for c in [1, 2]:  # class 2,3 (หลัง shift)
    if before.get(c, 0) < target_size:
        sampling_strategy[c] = target_size

if sampling_strategy:
    smt = SMOTETomek(
        sampling_strategy=sampling_strategy,
        random_state=42
    )
    X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
    after = Counter(y_train_res)
else:
    X_train_res, y_train_res = X_train, y_train
    after = before

print("After SMOTE+Tomek (train):", after)


Before SMOTE+Tomek (train): Counter({np.int64(4): 128801, np.int64(3): 73062, np.int64(0): 64314, np.int64(1): 33504, np.int64(2): 29942})
After SMOTE+Tomek (train): Counter({np.int64(4): 123154, np.int64(3): 65308, np.int64(2): 61602, np.int64(0): 61200, np.int64(1): 60905})


In [4]:
# ---------------- 3) LSTM Model Definition ----------------
class SimpleLSTM(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(SimpleLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,   # จำนวน features
            hidden_size=64,         # units = 64 (default style)
            num_layers=1,           # 1 ชั้น
            batch_first=True,       # (batch, seq, feature)
            dropout=0.0,            # ไม่มี dropout ภายใน LSTM
            bias=True
        )
        self.fc = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        # จาก CNN → ตอนนี้ x.shape = (batch, 1, feature)
        # ต้องแปลงเป็น (batch, seq_len, feature_dim)
        # ให้ seq_len = 1 → (batch, seq_len=1, feature_dim)
        x = x.squeeze(1).unsqueeze(1)
        out, _ = self.lstm(x)
        out = out[:, -1, :]           # เอาค่า timestep สุดท้าย
        out = self.dropout(out)
        out = self.fc(out)
        return out

print("\n=== LSTM Default Parameters ===")
lstm_test = nn.LSTM(
    input_size=8,   # example
    hidden_size=64,
    num_layers=1,
    batch_first=True,
    bias=True,
    dropout=0.0
)
print(lstm_test)



=== LSTM Default Parameters ===
LSTM(8, 64, batch_first=True)


In [5]:
# ---------------- 4) Utility Function ----------------
def train_one_fold(model, optimizer, criterion, loader, device):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()

In [6]:
# ---------------- 5) 10-Fold Cross Validation ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(np.unique(y_train_res))

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

val_preds, val_probas, val_true = [], [], []

for train_idx, val_idx in skf.split(X_train_res, y_train_res):
    X_tr, X_val = X_train_res[train_idx], X_train_res[val_idx]
    y_tr, y_val = y_train_res[train_idx], y_train_res[val_idx]

    X_tr_t = torch.tensor(X_tr).unsqueeze(1)
    y_tr_t = torch.tensor(y_tr)
    X_val_t = torch.tensor(X_val).unsqueeze(1)
    y_val_t = torch.tensor(y_val)

    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

    model_cv = SimpleLSTM(X_tr.shape[1], num_classes).to(device)
    optimizer = torch.optim.Adam(model_cv.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(5):
        train_one_fold(model_cv, optimizer, criterion, train_loader, device)

    model_cv.eval()
    with torch.no_grad():
        out_val = model_cv(X_val_t.to(device))
        y_val_pred = out_val.argmax(dim=1).cpu().numpy()
        y_val_proba = F.softmax(out_val, dim=1).cpu().numpy()

    val_preds.extend(y_val_pred)
    val_probas.extend(y_val_proba)
    val_true.extend(y_val)

# Metrics
acc_val = accuracy_score(val_true, val_preds)
p_val, r_val, f1_val, _ = precision_recall_fscore_support(val_true, val_preds, average="weighted")
roc_val = roc_auc_score(
    label_binarize(val_true, classes=np.unique(y_train_res)),
    np.array(val_probas),
    multi_class="ovr",
    average="macro"
)

validation_table = pd.DataFrame([{
    "Accuracy": acc_val,
    "Precision": p_val,
    "Recall": r_val,
    "F1-score": f1_val,
    "ROC-AUC": roc_val
}])
print("\nValidation Table:\n", validation_table)


Validation Table:
    Accuracy  Precision    Recall  F1-score   ROC-AUC
0  0.672807   0.666161  0.672807   0.66899  0.904234


In [7]:
# ---------------- 6) Train Full Model & Evaluate on Test ----------------
X_train_res_t = torch.tensor(X_train_res).unsqueeze(1)
y_train_res_t = torch.tensor(y_train_res)
train_ds_full = TensorDataset(X_train_res_t, y_train_res_t)
train_loader_full = DataLoader(train_ds_full, batch_size=256, shuffle=True)

cnn_model = SimpleLSTM(X_train_res.shape[1], num_classes).to(device)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(5):
    train_one_fold(cnn_model, optimizer, criterion, train_loader_full, device)

cnn_model.eval()
X_test_t = torch.tensor(X_test).unsqueeze(1).to(device)
with torch.no_grad():
    out_test = cnn_model(X_test_t)
    y_test_pred = out_test.argmax(dim=1).cpu().numpy()
    y_test_proba = F.softmax(out_test, dim=1).cpu().numpy()

acc_test = accuracy_score(y_test, y_test_pred)
p_test, r_test, f1_test, _ = precision_recall_fscore_support(y_test, y_test_pred, average="weighted")
roc_test = roc_auc_score(
    label_binarize(y_test, classes=np.unique(y_train_res)),
    y_test_proba,
    multi_class="ovr",
    average="macro"
)

test_table = pd.DataFrame([{
    "Accuracy": acc_test,
    "Precision": p_test,
    "Recall": r_test,
    "F1-score": f1_test,
    "ROC-AUC": roc_test
}])
print("\nTest Table:\n", test_table)


Test Table:
    Accuracy  Precision    Recall  F1-score   ROC-AUC
0  0.707461   0.709693  0.707461  0.706079  0.911485


In [8]:
# ---------------- 7) Save Results ----------------
validation_table.to_csv("Tomek_validation_metrics_LSTM.csv", index=False)
test_table.to_csv("Tomek_test_metrics_LSTM.csv", index=False)

proba_df = pd.DataFrame(y_test_proba, columns=[f"prob_{c}" for c in np.unique(y_train_res)])
proba_df.insert(0, "true", y_test)
proba_df.to_csv("Tomek_proba_test_LSTM.csv", index=False)

print("\nSaved: BSvalidation_metrics_cnn.csv, BStest_metrics_cnn.csv, BSproba_test_cnn.csv")


Saved: BSvalidation_metrics_cnn.csv, BStest_metrics_cnn.csv, BSproba_test_cnn.csv
